# 🏍️ R&V Motor - Prototipo del Agente Clasificador Inteligente
Este cuaderno contiene el prototipo de clasificación semántica de leads utilizando la API oficial de Gemini.

In [ ]:
# 1. Instalar SDK oficial de GenAI
!pip install -q google-genai pydantic pandas

In [ ]:
# 2. Configurar el esquema estricto y el cliente
from google import genai
from google.colab import userdata
from pydantic import BaseModel, Field
from typing import Literal

class AnalisisInteraccion(BaseModel):
    clasificacion: Literal['Lead Caliente', 'Cliente Potencial', 'Cliente que Solo Pregunta', 'Lead Frío'] = Field(
        description='Categoría de intención de compra del cliente de acuerdo a las reglas de negocio.'
    )
    ingreso_estimado: float = Field(
        description='Ingreso estimado basado en la categoría (Leads Calientes = 2500, Potenciales = 1200, otros = 0).'
    )
    roi_estimado: float = Field(
        description='Retorno de Inversión estimado porcentual (ej: 1566% expresado como 1566.0).'
    )
    respuesta_automatica: str = Field(
        description='Respuesta persuasiva, humana y empática en español para WhatsApp, orientada a cerrar la venta.'
    )

try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=GEMINI_API_KEY)
    print('¡Cliente de Gemini inicializado con éxito!')
except Exception as e:
    print(f'Error al cargar la API Key de Gemini: {e}')

In [ ]:
# 3. Función del Agente de Inteligencia Artificial
def clasificar_mensaje(mensaje_cliente: str) -> AnalisisInteraccion:
    system_prompt = '''
    Actúas como el Agente de IA experto de la concesionaria de motocicletas "R&V Motor".
    Analiza el contexto del mensaje enviado por el cliente en menos de 3 segundos y clasifícalo en una de estas categorías obligatorias:

    1. 'Lead Caliente': Intención clara e inmediata de compra (precios específicos, cuotas, financiamiento, métodos de pago, quiere ir hoy mismo con el dinero).
       - Métrica financiera asociada: ingreso_estimado = 2500.00, roi_estimado = 1566.0
    2. 'Cliente Potencial': Interés real pero con planes a futuro (agendar citas próxima semana, disponibilidad próximo mes, catálogos generales).
       - Métrica financiera asociada: ingreso_estimado = 1200.00, roi_estimado = 750.0
    3. 'Cliente que Solo Pregunta': Soporte o información general operativa sin venta directa (horarios, dirección física, repuestos, vacantes).
       - Métrica financiera asociada: ingreso_estimado = 0.0, roi_estimado = 0.0
    4. 'Lead Frío': Negativa, descartes ("No gracias"), emoticones por error o textos sin sentido ajenos a motos.
       - Métrica financiera asociada: ingreso_estimado = 0.0, roi_estimado = 0.0

    REGLA DE RESPUESTA AUTOMÁTICA: Genera una respuesta persuasiva, humana y empática en español, orientada a cerrar la venta o resolver el soporte.
    '''

    response = client.models.generate_content(
        model='gemini-3.8-flash',
        contents=mensaje_cliente,
        config=genai.types.GenerateContentConfig(
            system_instruction=system_prompt,
            response_mime_type='application/json',
            response_schema=AnalisisInteraccion,
            temperature=0.2
        )
    )
    return response.parsed

# Prueba rápida
mensaje_test = 'Hola! Me interesa muchísimo la promoción de la Yamaha FZ-S a US$2,999. ¿Tienen planes de financiamiento?'
try:
    print(clasificar_mensaje(mensaje_test))
except Exception as e:
    print(e)